# Ray 2.60：LLM 后训练实战入门

本 Notebook 用 Ray Train 在多 GPU 上做监督微调（SFT），重点讲清楚 Ray 如何负责分布式调度、数据分片、故障恢复和 checkpoint。后续章节说明如何扩展到 LoRA/QLoRA、DPO 与 GRPO。

**适用环境：** Linux + NVIDIA GPU（建议单机 1 张 24 GB 或更多显存的 GPU 起步；多卡需要 NCCL）。示例模型为 `Qwen/Qwen2.5-0.5B-Instruct`，实际模型需检查 Hugging Face 访问权限、许可证和显存。Windows 原生环境不建议直接跑多 GPU 训练，请使用 Linux 或 WSL2。

> 文档核对日期：2026-11-17。Ray 官方文档与 PyPI 当前版本为 2.60.0。Ray 版本迭代较快，升级版本时请回看官方 API。

## 1. 后训练与 Ray 组件

SFT 使用问答/指令样本优化 next-token loss。DPO 使用偏好对直接优化策略；GRPO/PPO 则需要 rollout、奖励模型或规则奖励。Ray 的角色是把工作拆成可调度的分布式任务：Ray Data 负责数据流水线，Ray Train 负责多 worker 训练，Ray Tune 可做超参搜索；在线推理可以另接 Ray Serve。

本教程先完成可检查的 SFT 工作流。真正的大规模偏好优化通常会用专门的训练框架（例如 verl、TRL 或 OpenRLHF），并由 Ray 集群承载其数据/训练/rollout 作业。

In [ ]:
# 先按 https://pytorch.org/get-started/locally/ 安装匹配 CUDA 的 PyTorch。
# 下面是教程选定的依赖组合，尚未在本机完成 GPU 集成验证；不要盲目升级所有包。
%pip install "ray[train,data,default]==2.60.0" "transformers==4.57.1" "datasets==4.2.0" "accelerate==1.10.1" "peft==0.17.1" safetensors
# 安装后重启 kernel；成功训练后使用 pip freeze 保存实际环境。

In [ ]:
import os
import ray
import torch

print("Ray:", ray.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. 启动 Ray 与集群检查

本地学习可以 `ray.init()`；生产多机环境应连接已经启动的 Ray cluster（head/worker 节点版本、Python 环境和依赖必须一致）。训练 worker 所需的软件可以用 runtime environment 安装，但大模型集群通常会预先构建一致的容器镜像。

In [ ]:
if not ray.is_initialized():
    ray.init()  # 已有集群时改为 ray.init(address="auto")

print(ray.cluster_resources())
print(ray.available_resources())

## 2.1 先用 CPU 理解 Ray：Task、ObjectRef、Actor

Task 适合独立计算，例如每个回答的规则奖励；Actor 是持有状态的进程，例如常驻 GPU 的 rollout 模型。
`.remote()` 立即返回 ObjectRef，`ray.get()` 等待结果。不要在提交循环里逐个 get，否则会串行等待。
下面仅计算玩具规则奖励，没有调用 LLM，也不是完整 RL 训练。

In [ ]:
@ray.remote(num_cpus=1)
def rule_reward(answer, expected):
    return float(answer.strip() == expected)

refs = [rule_reward.remote(answer, "4") for answer in ["4", "5", " 4 "]]
print("parallel rewards:", ray.get(refs))

@ray.remote(num_cpus=1)
class RewardCounter:
    def __init__(self):
        self.count = 0
    def record(self, rewards):
        self.count += len(rewards)
        return {"samples": self.count, "mean_reward": sum(rewards) / len(rewards)}

counter = RewardCounter.remote()
print(ray.get(counter.record.remote(ray.get(refs))))
ray.kill(counter)  # 释放 Actor 的 CPU 资源

**资源不是内存配额。** `num_gpus=1` 是调度声明，不能自动限制显存；fractional GPU 也不会隔离显存。
GPU Actor 的模型应在 `__init__` 加载一次，推理多次复用。训练和 rollout 同时占满 GPU 会导致等待甚至资源死锁。
大规模任务使用 `ray.wait()` 限制在途 ObjectRef 数量，避免把所有生成结果同时堆进 object store。

## 3. 构造一份小型指令数据

先用少量合成样本验证代码链路，不能用它衡量模型效果。生产训练时把数据换成经过授权、去重、清洗和脱敏的语料；至少做训练/验证切分，并检查格式、长度分布、重复率和隐私信息。聊天模板必须与目标模型匹配。

In [ ]:
from datasets import Dataset

# 按独立问题划分，再重复训练样本，避免同一个问题泄漏进验证集。
examples = [
    {"instruction": "用一句话解释 Ray。", "response": "Ray 是用于分布式 Python 计算的框架。"},
    {"instruction": "什么是 SFT？", "response": "SFT 使用带目标答案的数据继续训练模型。"},
    {"instruction": "Python 如何反转列表？", "response": "可以使用 items[::-1]。"},
    {"instruction": "checkpoint 有什么作用？", "response": "它保存模型及训练状态，用于评估或恢复。"},
]
train_ds = {
    "train": Dataset.from_list(examples[:3] * 32),
    "test": Dataset.from_list(examples[3:]),
}
train_ds

## 4. 使用 Ray Data 做可扩展预处理

Ray Data 适合将大规模文件读取、过滤、转换、分片等操作并行化。下面把 Hugging Face Dataset 转换为 Ray Dataset，并用模型 chat template 形成训练文本。简单示例中 tokenizer 在训练 worker 内调用；大规模训练应基于吞吐、CPU/GPU 利用率和缓存策略决定是否预先 tokenize。

In [ ]:
from transformers import AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
# 正式实验填模型 commit hash，避免 main 分支变化。
MODEL_REVISION = "main"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def to_chat_text(row):
    messages = [
        {"role": "user", "content": row["instruction"]},
        {"role": "assistant", "content": row["response"]},
    ]
    return {"text": tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )}

ray_train_ds = ray.data.from_huggingface(train_ds["train"]).map(to_chat_text)
ray_valid_ds = ray.data.from_huggingface(train_ds["test"]).map(to_chat_text)
ray_train_ds.take(2)

## 5. Ray Train + LoRA SFT

默认采用 LoRA，只训练注意力投影上的 adapter。设置 `use_lora=False` 才是全量微调。
Ray 启动每卡一个进程，Transformers/Accelerate 建立 DDP，Trainer 对每个 worker 中相同的 HF Dataset 做分布式采样。
**这里不能再手动 Ray 分片，否则会发生重复分片。**

为降低集成复杂度，小数据教程用 `take_all()` 收集处理结果；生产数据不要照搬。大数据改用 Ray Train `datasets` / `get_dataset_shard()` 配合自定义 DataLoader，或者让每个 worker 读取同一份磁盘 HF Dataset，由 Trainer 分布式采样。

当前 loss 覆盖完整聊天序列（含 user tokens），属于教学简化。真实指令 SFT 常只监督 assistant token；需要按模型模板生成 label mask，并逐条核对。不能仅按字符串位置猜测 token 边界。

有效全局 batch = 每卡 batch × 梯度累积步数 × worker 数。DDP 每卡仍持有完整基础模型；LoRA 不会分片模型权重。

In [ ]:
import tempfile
from pathlib import Path
from ray import train
from ray.train import Checkpoint, RunConfig, ScalingConfig
from ray.train.torch import TorchTrainer
from ray.train.huggingface.transformers import prepare_trainer

# 仅小数据演示允许收集到 driver。
train_rows = ray_train_ds.take_all()
eval_rows = ray_valid_ds.take_all()
MAX_LENGTH = 256
assert train_rows and eval_rows
print("train/eval rows:", len(train_rows), len(eval_rows))

In [ ]:
def train_loop_per_worker(config):
    import torch
    from datasets import Dataset
    from transformers import (
        AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments,
        DataCollatorForSeq2Seq, set_seed,
    )
    from peft import LoraConfig, get_peft_model

    set_seed(42)
    context = train.get_context()
    tok = AutoTokenizer.from_pretrained(config["model_id"], revision=config["revision"])
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "right"
    use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    model = AutoModelForCausalLM.from_pretrained(
        config["model_id"], revision=config["revision"],
        torch_dtype=torch.bfloat16 if use_bf16 else torch.float32,
    )
    model.config.use_cache = False
    if config["use_lora"]:
        model = get_peft_model(model, LoraConfig(
            task_type="CAUSAL_LM", r=8, lora_alpha=16, lora_dropout=0.05,
            target_modules=["q_proj", "v_proj"],
        ))
        model.print_trainable_parameters()

    def tokenize(batch):
        encoded = tok(batch["text"], truncation=True,
                      max_length=config["max_length"], add_special_tokens=False)
        encoded["labels"] = [ids.copy() for ids in encoded["input_ids"]]
        return encoded

    def make_dataset(rows):
        return Dataset.from_list(rows).map(tokenize, batched=True, remove_columns=["text"])

    # 只把 padding 的 labels 设为 -100；保留真实 EOS 的监督信号。
    collator = DataCollatorForSeq2Seq(tok, label_pad_token_id=-100, pad_to_multiple_of=8)
    with tempfile.TemporaryDirectory(prefix="ray-sft-") as workdir:
        args = TrainingArguments(
            output_dir=workdir, max_steps=config["max_steps"],
            per_device_train_batch_size=config["batch_size"],
            per_device_eval_batch_size=1,
            gradient_accumulation_steps=config["grad_accum"],
            learning_rate=config["learning_rate"],
            logging_steps=1, save_strategy="no", eval_strategy="no",
            bf16=use_bf16, fp16=False,
            gradient_checkpointing=True,
            gradient_checkpointing_kwargs={"use_reentrant": False},
            ddp_find_unused_parameters=False, report_to=[], seed=42,
        )
        hf_trainer = Trainer(
            model=model, args=args, data_collator=collator,
            train_dataset=make_dataset(config["train_rows"]),
            eval_dataset=make_dataset(config["eval_rows"]),
        )
        hf_trainer = prepare_trainer(hf_trainer)
        output = hf_trainer.train()
        metrics = hf_trainer.evaluate()
        metrics["train_loss"] = output.training_loss
        checkpoint = None
        if context.get_world_rank() == 0:
            export = Path(workdir) / "export"
            hf_trainer.save_model(str(export))
            tok.save_pretrained(str(export))
            checkpoint = Checkpoint.from_directory(str(export))
        # 所有 rank 调用 report 相同次数；只由 rank 0 上传模型文件。
        train.report(metrics, checkpoint=checkpoint)

## 6. 启动训练

这是 10 个 optimizer steps 的 smoke run，目的在于检查工程链路，不能证明模型质量提高。
首次运行会下载模型。修改下方 `RUN_TRAINING=True` 才会实际启动。

默认 LoRA 学习率为 2e-4；全量微调需要另行选择（例如从 2e-5 开始试验）。
`storage_path` 本地目录只适合单机；多机必须换成所有节点可访问的持久存储。
Notebook 默认不自动开启重试，避免把 GPU OOM 反复重启成无效作业。

In [ ]:
RUN_TRAINING = False
NUM_WORKERS = 1
USE_LORA = True
result = None
if RUN_TRAINING:
    assert torch.cuda.is_available(), "请在具备 CUDA 的 Linux/WSL2 环境运行"
    assert ray.cluster_resources().get("GPU", 0) >= NUM_WORKERS
    trainer = TorchTrainer(
        train_loop_per_worker,
        train_loop_config={
            "model_id": MODEL_ID, "revision": MODEL_REVISION,
            "max_steps": 10, "max_length": MAX_LENGTH,
            "batch_size": 1, "grad_accum": 4,
            "learning_rate": 2e-4 if USE_LORA else 2e-5,
            "use_lora": USE_LORA,
            "train_rows": train_rows, "eval_rows": eval_rows,
        },
        scaling_config=ScalingConfig(
            num_workers=NUM_WORKERS, use_gpu=True,
            resources_per_worker={"CPU": 1, "GPU": 1},
        ),
        run_config=RunConfig(
            name="ray-llm-sft", storage_path=str(Path("ray_results").resolve()),
        ),
    )
    result = trainer.fit()
    print(result.metrics)
    print(result.checkpoint)
else:
    print("已跳过 GPU 训练；准备好环境后设置 RUN_TRAINING=True。")

## 7. 加载 adapter / 全量模型做生成检查

LoRA checkpoint 仅保存 adapter 和 tokenizer，推理时还必须加载原基础模型。下面在 CPU 上生成，速度较慢；有空闲 GPU 可把模型移到 CUDA。
验证集只有一个独立问题，不能作为可信 benchmark。正式评估需固定任务集、模型版本和解码参数，比较基础模型与微调模型的正确率、格式遵循和安全表现。

**恢复边界：** 本示例导出的是模型/adapter 权重，不含 optimizer、scheduler 和 RNG 状态，不能精确续训。精确恢复应让 HF Trainer 保存完整训练 checkpoint，并通过 Ray checkpoint 传递，再在 worker 内使用 `trainer.train(resume_from_checkpoint=...)`；各 rank 必须恢复同一训练步。

In [ ]:
if result is not None:
    from transformers import AutoModelForCausalLM
    from peft import PeftModel
    with result.checkpoint.as_directory() as checkpoint_dir:
        tok = AutoTokenizer.from_pretrained(checkpoint_dir)
        if USE_LORA:
            base = AutoModelForCausalLM.from_pretrained(
                MODEL_ID, revision=MODEL_REVISION, torch_dtype=torch.float32)
            model = PeftModel.from_pretrained(base, checkpoint_dir)
        else:
            model = AutoModelForCausalLM.from_pretrained(
                checkpoint_dir, torch_dtype=torch.float32)
        model.eval()
        messages = [{"role": "user", "content": "checkpoint 有什么作用？"}]
        prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tok(prompt, return_tensors="pt", add_special_tokens=False)
        with torch.inference_mode():
            out = model.generate(**inputs, max_new_tokens=64, do_sample=False,
                                 pad_token_id=tok.pad_token_id)
        print(tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))
else:
    print("训练后再运行此单元。")

## 8. 从单机扩到集群

- 启动 Ray head 和 worker 节点，确认 `ray status` 能看到预期 GPU；Notebook 连接 `ray.init(address="auto")`。
- `ScalingConfig(num_workers=N, use_gpu=True)` 控制训练进程数量。训练方式（DDP、FSDP、DeepSpeed）决定显存切分和通信特性；多机大模型通常需要 FSDP/DeepSpeed，而非朴素 DDP。
- `RunConfig(storage_path=...)` 指向集群所有节点可访问的持久存储。验证中断恢复、checkpoint 完整性和依赖一致性。
- 用 Ray Data 读取 Parquet/JSONL 等分片文件，过滤坏样本、控制序列长度并观察数据吞吐；避免所有数据先读入 Notebook driver 内存。
- 训练前测量 tokens/s、GPU 利用率、数据等待时间和通信开销；加卡不一定线性提速。

## 9. LoRA、DPO 与 GRPO 的后续路径

**LoRA/QLoRA：** 在模型加载后用 PEFT 注入 adapter，并只训练 adapter 参数，可大幅降低优化器状态和梯度显存；QLoRA 还需要兼容的 4-bit bitsandbytes 环境。必须验证 Ray worker 的 CUDA、bitsandbytes 和 PEFT 版本组合。

**DPO：** 将数据转成 `(prompt, chosen, rejected)` 偏好对，使用 TRL `DPOTrainer` 或其它支持的训练循环。Ray Train 负责 worker 调度和资源管理；要特别关注 reference model 的显存开销、序列长度、偏好数据质量。

**GRPO/PPO：** 除策略训练外，还要调度 rollout 推理与奖励计算。多组件资源编排、异步采样和 checkpoint 语义更复杂，建议采用专用框架（如 verl）并按其官方 Ray 部署说明配置，而不是把本 Notebook 的 SFT 循环直接改名为 RL。

生产化还需要：确定性数据版本、可恢复的持久 checkpoint、实验追踪、离线评测套件、模型卡/许可证核查、内容安全评估，以及训练/推理格式兼容性测试。

## 官方资料

- [Ray Train 文档](https://docs.ray.io/en/latest/train/train.html)
- [Ray Train + Hugging Face Transformers](https://docs.ray.io/en/latest/train/api/doc/ray.train.huggingface.transformers.prepare_trainer.html)
- [Ray Data 文档](https://docs.ray.io/en/latest/data/data.html)
- [Ray 文档示例集](https://docs.ray.io/en/latest/ray-overview/examples.html)
- [Ray GitHub](https://github.com/ray-project/ray)

本 Notebook 展示 Ray 训练编排的基础模式。Ray 2.60.0 的 API、Transformers 的 Trainer 参数名，以及集成支持范围可能随版本变化；开跑前建议对照上述当前文档。

## 10. DPO 与 GRPO：从数据接口理解下一步

| 路线 | 数据 | 核心额外资源 | 优先检查 |
|---|---|---|---|
| SFT | prompt + answer | policy | assistant label mask、模板 |
| DPO | prompt + chosen + rejected | policy、reference | pair 质量、reference 与 policy 起点 |
| GRPO | prompt、可验证答案/奖励函数 | policy、rollout，按配置可有 reference | 组内奖励差异、KL、长度偏置 |

DPO 对同一 prompt 下 chosen/rejected 的相对 log-prob 优化；不需要在线 rollout。
GRPO 对同一 prompt 采样一组答案，根据相对奖励构造 advantage，再更新 policy。
若所有奖励都相同，组内标准化后没有有效相对信号；不能只看平均 reward，要看组内标准差、KL、输出长度与外部评测。

Ray 在这里负责进程和资源编排，不会自动提供 DPO/GRPO 的正确损失函数或权重同步。
TRL 适合从单机算法实验开始；verl/OpenRLHF 等有各自的 Ray 编排和版本要求。安装这些框架时使用其锁定环境，不能假设与此 Notebook 的依赖组合兼容。

In [ ]:
# 偏好数据的最小格式；不是训练调用。
preference_pair = {
    "prompt": [{"role": "user", "content": "2+2 等于多少？"}],
    "chosen": [{"role": "assistant", "content": "4"}],
    "rejected": [{"role": "assistant", "content": "5"}],
}
assert preference_pair["chosen"] != preference_pair["rejected"]

# GRPO 相对奖励的教学计算；实际训练还需 policy loss、KL、clip 等。
rewards = torch.tensor([1., 0., 1., 0.])
advantages = (rewards - rewards.mean()) / (rewards.std(unbiased=False) + 1e-6)
print("group advantages:", advantages.tolist())

## 11. 排查问题与练习

| 现象 | 检查与处理 |
|---|---|
| worker 一直 PENDING | `ray.available_resources()`；CPU/GPU 是否被 Actor 占满 |
| CUDA OOM | 降低长度/batch，LoRA；DDP 不会自动分片基础权重 |
| 多机 NCCL 卡住 | 网络互通、各节点 CUDA/驱动、rank 是否进入相同通信步骤 |
| loss 为 NaN | 输入标签、学习率、精度与 bf16 硬件支持 |
| checkpoint 无法续训 | 区分 adapter 导出与包含 optimizer/RNG 的完整训练状态 |
| 数据越多 driver 越慢 | 不要生产使用 `take_all()`；改为分片存储与流式读取 |

练习：① 改为两卡并计算有效 batch；② 固定全局 batch 比较吞吐；③ 为三条样本打印 token 与 label mask；④ 替换成独立验证集；⑤ 人为中断一个完整 checkpoint 训练并验证恢复步数。

当前 Notebook 未执行 GPU 训练，不包含伪造 loss、吞吐或效果指标。语法检查不能代替硬件环境下的集成测试。

### 补充官方链接

- [Ray 2.60.0 发布代码](https://github.com/ray-project/ray/tree/ray-2.60.0)
- [Ray Core 概念](https://docs.ray.io/en/latest/ray-core/key-concepts.html)
- [Ray Hugging Face 集成源码](https://github.com/ray-project/ray/blob/ray-2.60.0/python/ray/train/huggingface/transformers/_transformers_utils.py)
- [TRL DPO](https://huggingface.co/docs/trl/dpo_trainer)
- [TRL GRPO](https://huggingface.co/docs/trl/grpo_trainer)
- [verl](https://github.com/volcengine/verl)
- [OpenRLHF](https://github.com/OpenRLHF/OpenRLHF)

“最新”指核对日期的 Ray 发布版本；本教程未声称使用最新模型或所有依赖的最新版本。

In [ ]:
# 完成本地学习后手动执行，释放本进程创建的 Ray runtime：
# ray.shutdown()